# Exploração dos dados brutos — Série histórica de preços da ANP

Este notebook faz um inventário inicial dos arquivos CSV baixados pelo pipeline
de coleta, para entender o volume de dados antes da análise.

## Imports

In [1]:
import pandas as pd
from pathlib import Path
import re

## Mapeamento dos arquivos

Os dados brutos ficam em `raw/`, organizados em uma subpasta por categoria de produto.
Aqui definimos:

- **`all_csvs`**: todos os arquivos CSV, buscando recursivamente em `raw/`
- **`months_dict`**: tabela de conversão do nome do mês (em minúsculas) para o seu número, usada para interpretar os nomes dos arquivos mensais

In [2]:
raw_path = Path('raw')

# Todos os arquivos
all_csvs = list(raw_path.rglob('*.csv'))

# Meses e número equivalentes
months_dict = {'janeiro': 1, 'fevereiro': 2, 'março': 3, 'abril': 4, 'maio': 5, 'junho': 6,
               'julho': 7, 'agosto': 8, 'setembro': 9, 'outubro': 10, 'novembro': 11, 'dezembro': 12}

## Função auxiliar de formatação

Converte um tamanho em MB para um texto legível: exibe em **GB** quando passa
de 1024 MB e em **MB** caso contrário.

In [3]:
def format_mb(value):
    return f"{value/1024:.2f} GB" if value > 1024 else f"{value:.2f} MB"

## Inventário por arquivo

Monta uma tabela (`inventory`) com uma linha por arquivo, contendo:

- **categoria**: nome da pasta onde o arquivo está
- **arquivo** e **tamanho** em disco
- **periodo**: `Semestral` ou `Mensal`
- **inicio** e **fim**: o período que o arquivo **declara** cobrir, extraído do nome

Os nomes seguem dois padrões, interpretados com regex:

| Padrão | Exemplo | Período |
|---|---|---|
| semestral | `1º semestre de 2004.csv` | 2004-01-01 a 2004-06-30 |
| mensal | `Abril de 2023.csv` | 2023-04-01 a 2023-04-30 |

O padrão semestral usa `\D+` entre o número e a palavra "semestre" para aceitar tanto
`º` (indicador ordinal) quanto `°` (símbolo de grau): o arquivo `2° semestre de 2023.csv`
usa o segundo, diferente de todos os outros.

As datas de início e fim são o período **declarado** pelo nome. Na etapa de anomalias,
elas serão comparadas com as datas reais de coleta dentro de cada arquivo.

Os totais por categoria não são guardados na tabela: são calculados a partir dela com `groupby`.

> **Unidades:** os tamanhos usam base binária (1 MB = 1024² bytes,
> 1 GB = 1024 MB), a mesma convenção do Windows. Assim os valores batem
> com as propriedades da pasta no Explorador de Arquivos.

In [4]:
# Lista com as informações por arquivo
info_list = []

# Para todo o diretório raw
for csv in all_csvs:

    # Nome do arquivo csv
    archive_name = csv.name

    # Categoria do arquivo
    category = csv.parent.name

    # Busca regex
    m = re.search(r'(\d)\D+ semestre de (\d{4})', archive_name)

    # Se a busca encontrar o padrão
    if m is not None:

        # Semestre coberto pelo arquivo
        semester = int(m.group(1))

        # Ano dos dados do arquivo
        year = int(m.group(2))

        # Tipo do período
        period = 'Semestral'

        inicial_month = 1 if semester == 1 else 7

        # Data de ínicio da combertura
        start_period = pd.Timestamp(year=year, month=inicial_month, day=1)

        # Data de fim da cobertura
        finish_period = start_period + pd.DateOffset(months=6) - pd.Timedelta(days=1)

    else:

        # Faz a busca no segundo padrão
        m = re.search(r'(\w+) de (\d{4})', archive_name)

        # Mês de referência do arquivo
        month = m.group(1).lower()

        # Ano de refência do arquivo
        year = int(m.group(2))

        # Tipo do período
        period = 'Mensal'

        month_normalizad = months_dict.get(month)

        # Data de ínicio da combertura
        start_period = pd.Timestamp(year=year, month=month_normalizad, day=1)

        # Data de fim da cobertura
        finish_period = start_period + pd.DateOffset(months=1) - pd.Timedelta(days=1)

    # Tamanho do arquivo
    archive_size = csv.stat().st_size / 1024 ** 2

    info_list.append(
        {
            'categoria':category,
            'arquivo':archive_name,
            'tamanho':archive_size,
            'periodo': period,
            'inicio':start_period,
            'fim':finish_period
        }
    )

inventory = pd.DataFrame(info_list)

# Verificação de lacunas

In [5]:
# Dicionário para mapear o período para o pandas
freq = {'Mensal':'MS', 'Semestral':'6MS'}

# Lista com o resultado da verificação
gaps_list = []

# Itera sobre as categorias
for category, group in inventory.groupby('categoria'):
    
    # Pega o período usado na categoria
    period_type = group['periodo'].iloc[0]

    # Lista do intervalo de datas esperado
    expected = pd.date_range(group['inicio'].min(), group['inicio'].max(), freq=freq[period_type])

    # Diferença entre o esperado e o intervalo real
    missing = sorted(set(expected) - set(group['inicio']))

    gaps_list.append(
        {
            'categoria': category,
            'esperados': len(expected),
            'encontrados': len(group),
            'faltando': missing
        }
    )

gaps = pd.DataFrame(gaps_list)

## Resultado do inventário

Resumo por categoria: quantidade de arquivos, tamanho em disco e o intervalo coberto
(do início do primeiro período ao fim do último). Em seguida, o cruzamento entre
categoria e tipo de período, e a tabela completa ordenada cronologicamente. Verificação de lacunas nas datas cobertas.

In [6]:
print(f"Total de arquivos encontrados: {len(inventory)}")
print(f"Tamanho total dos arquivos: {format_mb(inventory['tamanho'].sum())}")

# Resumo por categoria
summary = inventory.groupby('categoria').agg(
    arquivos=('arquivo', 'count'),
    tamanho=('tamanho', 'sum'),
    primeiro=('inicio', 'min'),
    ultimo=('fim', 'max'),
)

print("\nResumo por categoria")

# Formata só na exibição: as colunas continuam numéricas (MB) e datas
display(summary.style.format({
    'tamanho': format_mb,
    'primeiro': '{:%Y-%m-%d}',
    'ultimo': '{:%Y-%m-%d}',
}))

print("\nContagem de tipo de período por categoria")

# Quantos arquivos de cada tipo de período existem em cada categoria
display(pd.crosstab(inventory['categoria'], inventory['periodo']))

print("\nVerificação de lacunas")

if gaps['faltando'].str.len().sum() == 0:
    print("Não foram encontradas lacunas nos períodos cobertos pelos arquivos")

else:
    display(gaps)

print("\nTabela completa com as informações coletadas")

# Tabela completa, em ordem cronológica dentro de cada categoria
display(inventory.sort_values(['categoria', 'inicio']).reset_index(drop=True))

Total de arquivos encontrados: 182
Tamanho total dos arquivos: 5.59 GB

Resumo por categoria


,arquivos,tamanho,primeiro,ultimo
categoria,,,,
Combustíveis automotivos,45,4.11 GB,2004-01-01,2026-06-30
Etanol Hidratado + Gasolina C,46,370.77 MB,2023-01-01,2026-10-31
GLP P13,45,969.64 MB,2004-01-01,2026-06-30
Óleo Diesel (S-500 e S-10) + GNV,46,170.62 MB,2023-01-01,2026-10-31



Contagem de tipo de período por categoria


periodo,Mensal,Semestral
categoria,,
Combustíveis automotivos,0,45
Etanol Hidratado + Gasolina C,46,0
GLP P13,0,45
Óleo Diesel (S-500 e S-10) + GNV,46,0



Verificação de lacunas
Não foram encontradas lacunas nos períodos cobertos pelos arquivos

Tabela completa com as informações coletadas


,categoria,arquivo,tamanho,periodo,inicio,fim
0,Combustíveis automotivos,1º semestre de 2004.csv,46.384674,Semestral,2004-01-01,2004-06-30
1,Combustíveis automotivos,2º semestre de 2004.csv,150.785591,Semestral,2004-07-01,2004-12-31
2,Combustíveis automotivos,1º semestre de 2005.csv,150.430410,Semestral,2005-01-01,2005-06-30
3,Combustíveis automotivos,2º semestre de 2005.csv,121.860354,Semestral,2005-07-01,2005-12-31
4,Combustíveis automotivos,1º semestre de 2006.csv,143.310256,Semestral,2006-01-01,2006-06-30
...,...,...,...,...,...,...
177,Óleo Diesel (S-500 e S-10) + GNV,Junho de 2026.csv,3.954677,Mensal,2026-06-01,2026-06-30
178,Óleo Diesel (S-500 e S-10) + GNV,Julho de 2026.csv,3.663260,Mensal,2026-07-01,2026-07-31
179,Óleo Diesel (S-500 e S-10) + GNV,Agosto de 2026.csv,3.877467,Mensal,2026-08-01,2026-08-31
180,Óleo Diesel (S-500 e S-10) + GNV,Setembro de 2026.csv,3.780302,Mensal,2026-09-01,2026-09-30
